In [ ]:
!pip install emoji

In [ ]:
#những thư viện cơ bản
import numpy as np
import nltk
import pandas as pd
import matplotlib.pyplot as plt
import re
#kiểm tra emoji
import emoji

#ttsongsong
from multiprocessing import Pool, cpu_count
from tqdm import tqdm



#những thư viện liên quan đến scikit learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import LogisticRegression

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')

In [ ]:
def label_to_num(label):
    if label.lower() == 'spam':
        return 1
    else:
        return 0


In [ ]:
test_pq = pd.read_parquet('test-00000-of-00001-fa9b3e8ade89a333.parquet')
train_pq = pd.read_parquet('train-00000-of-00001-daf190ce720b3dbb.parquet')
pq = pd.concat((train_pq, test_pq), ignore_index=True)
df = pd.read_csv('df.csv')
enron_data =  pd.read_csv('enron_spam_data.csv')
csa = pd.read_csv('completeSpamAssassin.csv')
#

In [ ]:
data = [pq, enron_data, df, csa]
name = ['pq', 'enron_data', 'df', 'csa']

In [ ]:
for i, d in enumerate(data):
    d.drop_duplicates(inplace=True)
    d.dropna(inplace=True)
    d.reset_index(drop=True, inplace=True)

In [ ]:

for i, d in enumerate(data):
    print(name[i])
    print(d.head())
    print(d.shape)
    print(d.info())
    print()

In [ ]:
for i, d in enumerate(data):
    if d.columns[0] == 'Unnamed: 0':
        d = d.drop(d.columns[0], axis=1, inplace=True)
enron_data = enron_data.drop(columns=['Date'])



In [ ]:
enron_data['text'] = enron_data['Subject'] + ' ' + enron_data['Message']
enron_data = enron_data.drop(columns=['Subject', 'Message'])
enron_data['label'] = enron_data['Spam/Ham'].apply(label_to_num)
enron_data = enron_data.drop(columns=['Spam/Ham'])
pq['label'] = pq['label'].apply(label_to_num)
csa = csa[csa['Body'] != 'empty']


In [ ]:
csa['text'] = csa['Body'].copy()
csa.drop(columns=['Body'], inplace=True)
csa['label'] = csa['Label'].copy()
csa.drop(columns=['Label'], inplace=True)


In [ ]:
data = [pq, enron_data, df, csa]
name = ['pq', 'enron_data', 'df', 'csa']

In [ ]:
norm = lambda s: re.sub(r'[^a-z]+', ' ', str(s).lower()).strip()[:500]
df_hash = set(df['text'].map(norm))

In [ ]:
for i, d in enumerate(data):
    if d is df:
        continue
    mask = d['text'].map(norm).isin(df_hash)
    print("dropping", mask.sum(), "of", len(d))
    d = d[~mask]

In [ ]:
# Gộp tất cả các DataFrame lại thành một tập dữ liệu chung
full_df = pd.concat([pq[['text', 'label']], enron_data[['text', 'label']], df[['text', 'label']], csa[['text', 'label']]], ignore_index=True)

# Loại bỏ các dòng trùng lặp có thể phát sinh sau khi gộp
full_df.drop_duplicates(inplace=True)
full_df.reset_index(drop=True, inplace=True)

# Kiểm tra thông tin tổng quan
print("Kích thước tập dữ liệu sau khi gộp:", full_df.shape)
display(full_df.head())


In [ ]:
# Thống kê tỷ lệ nhãn Spam (1) và Ham (0)
label_counts = full_df['label'].value_counts()
print("Phân bổ nhãn:")
print(label_counts)

# Vẽ biểu đồ trực quan hóa
fig = plt.figure(figsize=(6, 4))
label_counts.plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Phân bổ nhãn trong tập dữ liệu gộp')
plt.xlabel('Nhãn (0: Ham, 1: Phishing, 2: Spam )')
plt.ylabel('Số lượng mẫu')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:

# Khởi tạo bộ Stemmer để chuẩn hóa từ về dạng gốc
stemmer = nltk.stem.PorterStemmer()

def remove_subject(text):
    return re.sub(r'^Subject:', '', text)

def clean_emoji(text):
    for char in text:
        if emoji.is_emoji(char):
            text = text.replace(char, ' EMOJI ')
    return text

def clean_url(text):
    urls = re.findall(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', text)
    for url in urls:
        text = text.replace(url,  f'URL{len(url)}')
    return text

def clean_html(text):
    html_tags = re.findall(r'<[^>]+>', text)
    for tag in html_tags:
        text = text.replace(tag, ' HTML ')
    return text

def clean_number(text):
    numbers = re.findall(r'\d+', text)
    for number in numbers:
        text = text.replace(number, f' NUMBER{len(number)} ')
    return text

def clean_email_address(text):
    email_addresses = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',text)
    for email in email_addresses:
        text = text.replace(email, ' EMAIL ')
    return text

def clean_special_characters(text):
    # Giữ chữ cái, khoảng trắng và dấu nháy trong từ
    text = re.sub(r"[^a-zA-Z\s']", ' ', text)
    return text

def normalize_word(text):
    tmp = text.split()
    return ' '.join(stemmer.stem(word) for word in tmp)

def remove_stopwords(text):
    stop_words = set(nltk.corpus.stopwords.words('english'))
    tmp = text.split()
    return ' '.join(word for word in tmp if word not in stop_words)

def clean_text(text):
    text = text.lower()
    text = remove_subject(text)
    text = clean_email_address(text)
    text = clean_url(text)
    # text = clean_html(text)
    text = clean_emoji(text)
    text = clean_number(text)

    #text = normalize_word(text)
    # text = remove_stopwords(text)
    # text = clean_special_characters(text)
    text = re.sub(r'\s+', ' ', text)
    return text

test_text = train_pq['text'][12]
print("Văn bản gốc:")
print(test_text)
print("\nVăn bản sau khi làm sạch:")
print(clean_text(test_text))

In [ ]:
# from sklearn.pipeline import Pipeline

# # Khởi tạo Pipeline tích hợp cả Vectorizer và Model
# pipeline = Pipeline([
#     ('tfidf', TfidfVectorizer()),
#     ('lr', LogisticRegression())
# ])

# Thiết lập param_grid chung (sử dụng tiền tố tên bước + '__')
# param_grid = [
#     {
#         # TF-IDF
#         'tfidf__lowercase': [True],
#         'tfidf__max_df': [0.85, 0.95],
#         'tfidf__min_df': [1, 2],
#         'tfidf__max_features': [10000],
#         'tfidf__ngram_range': [(1, 1), (1, 2)],
#         'tfidf__sublinear_tf': [True],

#         # Logistic Regression
#         'lr__penalty': ['l2'],
#         'lr__solver': ['lbfgs'],
#         'lr__class_weight': [None, 'balanced'],
#     }
# ]

# # Khởi tạo một GridSearchCV duy nhất
# g = GridSearchCV(
#     estimator=pipeline,
#     param_grid=param_grid,
#     cv=3,
#     scoring='recall_macro',
#     n_jobs=-1,
#     verbose=2,
#     error_score='raise'
# )


In [ ]:
texts = full_df['text'].tolist()
#parallel processing
order = np.random.RandomState(0).permutation(len(texts))
shuffled = [texts[i] for i in order]

with Pool(4, maxtasksperchild=50) as pool:
    res = list(tqdm(pool.imap(clean_text, shuffled, chunksize=200), total=len(shuffled)))

cleaned = [None] * len(texts)
for pos, i in enumerate(order):
    cleaned[i] = res[pos]

full_df['text'] = cleaned
full_df.to_parquet('full_df.parquet')

#lấy data đã preprocess, vì quá lâu
full_df = pd.read_parquet('full_df.parquet')


In [ ]:
plt.figure(figsize=(8, 6))
full_df['label'].value_counts().plot(kind='bar', color=['skyblue', 'salmon'])

In [ ]:
X = full_df['text']
y = full_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2910)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
#g.fit(X_train, y_train)
#it took 3 hours to run
#here the stats:
#TfidfVectorizer(max_df=0.85, max_features=10000, ngram_range=(1, 2),sublinear_tf=True)
#LogisticRegression(class_weight='balanced')

In [ ]:
#vec, model = g.best_estimator_
vec = TfidfVectorizer(max_df=0.85, max_features=10000, ngram_range=(1, 2),
                      sublinear_tf=True)
model = LogisticRegression(class_weight='balanced')

X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

model.fit(X_train_vec, y_train)

In [ ]:
def get_top_tfidf_words(vectorizer, tfidf_matrix, top_n=20):
    # Lấy danh sách từ vựng từ vectorizer
    feature_names = np.array(vectorizer.get_feature_names_out())

    # Tính trung bình điểm TF-IDF của từng từ trên toàn bộ các dòng
    mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()

    # Sắp xếp chỉ số theo thứ tự giảm dần của điểm TF-IDF
    sorted_indices = np.argsort(mean_tfidf)[::-1]

    # Tạo DataFrame kết quả
    top_words_df = pd.DataFrame({
        'Từ/Cụm từ': feature_names[sorted_indices[:top_n]],
        'Điểm TF-IDF TB': mean_tfidf[sorted_indices[:top_n]]
    })
    return top_words_df

# Trích xuất top 20 từ có điểm TF-IDF cao nhất trên tập huấn luyện
top_tfidf = get_top_tfidf_words(vec, X_train_vec, top_n=20)
display(top_tfidf)

In [ ]:
y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))

In [ ]:
tester = '''
Dear People’s Bank Client,
For your security, the profi le that you are using to access People’s Bank Online Banking has been locked because
of too many failed login attempts. You can unlock this profi le online by selecting an option below:
Unlock your profi le with:
My ATM/Visa Check Card number and PIN.
Other personal information (Social Security Number, Account #, etc)
E-mail address.
We regret any inconvenience this may cause you.
Sincerely,
People’s Bank Account Review Department.
We are requesting this information to verify and protect your identity. This is in order to prevent the use of the U.S.
banking system in terrorist and other illegal activity.
Need help? Use “Site Helper” or call customer service at 1.800.788.7000.
Please do not “Reply” to this Alert.
©2005 People’s Bank Financial Group. All rights reserved
'''

model.predict_proba(vec.transform([tester]))


In [ ]:
import platform
import os
import psutil
import sklearn

print("Machine:", platform.machine())
print("CPU:", platform.processor())
print("Logical CPUs:", os.cpu_count())
print("RAM (GB):", round(
    psutil.virtual_memory().total / 1024**3, 2
))
print("scikit-learn:", sklearn.__version__)